# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library in Python.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Install `mlcroissant` if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"Dataset Title: {metadata['name']}\nDescription: {metadata['description']}")
print("Published Date:", metadata['datePublished'])
print("Authors (@id):", [a['@id'] for a in metadata.get('author', [])])

## 2. Data Overview
Let's review what record sets are available, and the field and column `@id`s that structure this dataset.

In [ ]:
# Get record sets from dataset metadata
record_sets = []
if 'recordSet' in metadata:
    record_sets = metadata['recordSet']
else:
    print('No recordSet found in metadata.')

print("Record Sets (@id):")
for rs in record_sets:
    if isinstance(rs, dict) and '@id' in rs:
        print(f"- {rs['@id']}")
    elif isinstance(rs, str):
        print(f"- {rs}")
    else:
        print(rs)

# If record sets are empty, try to discover via Croissant internal structure
if not record_sets:
    # mlcroissant exposes record set IDs via dataset.record_sets()
    from mlcroissant.structure import RecordSet
    record_sets = [r['@id'] for r in dataset.record_sets()]
    print("Discovered record sets:")
    for rs_id in record_sets:
        print(f"- {rs_id}")

# List fields for each record set
fields_ids = {}
for rs_id in record_sets:
    try:
        fields = dataset.field_ids(record_set=rs_id)
        fields_ids[rs_id] = fields
        print(f"\nRecordSet {rs_id} contains fields:")
        for field in fields:
            print(f"  - {field}")
    except Exception as e:
        print(f"Could not get fields for record set {rs_id}: {e}")

## 3. Data Extraction
Now, let's load data from each record set into Pandas DataFrames for analysis. All entities are referenced by their `@id`.

In [ ]:
# Prepare to load each record set as a DataFrame
dataframes = {}
loaded_record_sets = []

for rs_id in record_sets:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            loaded_record_sets.append(rs_id)
            print(f"\nLoaded record set {rs_id}: Columns: {list(df.columns)}")
            print(df.head())
        else:
            print(f"Record set {rs_id} contains no records.")
    except Exception as e:
        print(f"Failed to load {rs_id}: {e}")

# Choose one record set for demonstration (first loaded)
if loaded_record_sets:
    chosen_rs = loaded_record_sets[0]
    print(f"\nUsing record set: {chosen_rs}")
    df_demo = dataframes[chosen_rs]
    print(df_demo.head())
else:
    print("No record sets with records were loaded.")

## 4. Exploratory Data Analysis (EDA)
Let's demonstrate filtering numeric fields, normalization, and grouping. All operations reference columns by their `@id` from field lists above.

In [ ]:
# Demo: Pick a numeric field from the chosen record set
# For demonstration, try to find a numeric field automatically
numeric_field_id = None
if chosen_rs in fields_ids:
    # To select numeric, check for likely numeric columns
    candidate_fields = fields_ids[chosen_rs]
    for col in candidate_fields:
        if ('age' in col.lower()) or ('level' in col.lower()):
            numeric_field_id = col
            break

if numeric_field_id is None and df_demo.shape[1]:
    # Fallback: select first column
    numeric_field_id = df_demo.columns[0]

print(f"Selected numeric field for EDA: {numeric_field_id}")

# Filtering
threshold = 10
if numeric_field_id in df_demo.columns:
    filtered_df = df_demo[df_demo[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()

    print(f"Normalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field (if available)
    group_field_id = None
    for col in df_demo.columns:
        if ('phenotype' in col.lower()) or ('category' in col.lower()) or ('diagnosis' in col.lower()):
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"{numeric_field_id} not found in DataFrame columns.")

## 5. Visualization
Visualize the distribution and relationships between fields using Matplotlib and Seaborn.

In [ ]:
# Visualization of the numeric field distribution
if numeric_field_id and numeric_field_id in df_demo.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df_demo[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Visualize normalized values if available
    norm_col = f"{numeric_field_id}_normalized"
    if norm_col in filtered_df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(filtered_df[norm_col].dropna(), kde=True, color='orange')
        plt.title(f'Normalized {numeric_field_id} (filtered)')
        plt.xlabel(norm_col)
        plt.ylabel('Frequency')
        plt.show()

    # If grouping variable exists, plot group means
    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.show()

## 6. Conclusion
- This notebook introduced the FAIR^2 dataset and demonstrated how to access and analyze the structured clinical, laboratory, and genetic data for non-classical 21-hydroxylase deficiency-associated infertility using the `mlcroissant` library.
- By referencing entities via their `@id`, you can easily extract fields and perform filtering, normalization, or grouping tasks for research and clinical illustration.
- The dataset's small sample size and single-center data call for careful interpretation and extrapolation.

**For advanced exploration:**
- You can adapt this workflow to perform statistical correlations, visualization of genotype-phenotype relationships, or further medical analytics.
- For more information, consult the Croissant schema at the dataset URL above, or the [mlcroissant documentation](https://github.com/mlcommons/croissant).